# Ingest races.csv file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
   - Source File
   - Ingestion Timestamp
3. Write to bronze delta table  

In [0]:
%run "../00-common/01.environment-config"


In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
val source_file= landing_folder_path + "/races.csv"
val table_name= catalog_name + "." + bronze_schema + "." + "races"

In [0]:
import org.apache.spark.sql.types.{StructType,StructField,StringType,IntegerType,DateType}

val races_schema=StructType(Seq(
  StructField("season", IntegerType),
  StructField("round", IntegerType),
  StructField("url", StringType),
  StructField("raceName", StringType),
  StructField("date", DateType),
  StructField("circuitId", StringType),
))

val races_df=spark.read.format("csv")
.option("header","true")
.option("mode", "FAILFAST")
.schema(races_schema)
.load(source_file)

display(races_df)

In [0]:

val races_final_df= add_ingestion_metadata(races_df)

display(races_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
races_final_df.write.format("delta").mode("overwrite").saveAsTable(table_name)